In [1]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
import numpy as np
from tqdm import tqdm
import pickle
import csv
import sys
import matplotlib.pyplot as plt
from ordered_set import OrderedSet
import math
import copy
import re
from datetime import datetime
import numpy as np
import random

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#load the prefiltered documents list
with open("Data/comparing_contract_values.pickle", "rb") as f:
    comparing_contract_values = pickle.load(f)

#load the unique tags and frequencies list
with open("Data/Unique tags and frequencies.pickle", "rb") as f:
    sorted_tag_file_count = pickle.load(f)

#load the dictionary where you compare values of the contract
with open("Data/comparing_contract_values.pickle", "rb") as f:
    contract_values = pickle.load(f)

with open("Data/xml_files_list_2021.pickle", "rb") as f:
    xml_files_list_2020 = pickle.load(f)

with open("Data/xpath_expressions.pickle", "rb") as f:
    xpath_expressions = pickle.load(f)

with open('Data/pays_with_euro.pickle', "rb") as f:
    pays_with_euro = pickle.load(f)

with open("Data/val_total_col.pickle", "rb") as f:
    val_total_col = pickle.load(f)

with open("Data/country_to_currency_mapping.pickle", "rb") as f:
    country_to_currency_mapping = pickle.load(f)

with open("Data/no_money_values", "rb") as f:
    no_money_values = pickle.load(f)

with open("Data/indices_not_awarded", "rb") as f:
    indices_not_awarded = pickle.load(f)

with open("Data/data_dict_2020_ac_cost_trimmed", "rb") as f:
    list_of_dicts = pickle.load(f)
    

with open("Data/val_total_col.pickle", "rb") as f:
    val_total_col = pickle.load(f)

In [44]:
df = pd.read_pickle("Data/df_data_2021.pickle")

Methods to make the dataframe less memory intensive
1) when dropping use inplace = True to not produce a seperate daatframe each time

In [45]:
def get_column_groups(xpath_expressions: dict, selector: str):
    """This function takes in a dictionary of xpath expressions and return the expressions which result in multiple columns"""
    query_type_is_multiple = 2
    column_groups = []
    for expression in xpath_expressions:
        if xpath_expressions[expression][query_type_is_multiple] == selector:
            column_groups.append(expression)

    column_groups.remove("object_contract/val_total")

    return column_groups

In [46]:
def check_sparsity_dataframe(dataframe, column_groups):
    """This function takes in a dataframe and a set of columns, assesses the ratio of NaN or None values, and its increase with each additional column within that column group"""
    candidate_trim_columns = {}

    for column_group in column_groups:
        columns_within_group = []
        for column in dataframe.columns:
            if str(column_group + ":") in column:
                columns_within_group.append(column)
        candidate_trim_columns[column_group] = columns_within_group

    column_NaN_counts = {}
    
    for column_group in candidate_trim_columns.keys():
        NAN_counts = []
        for column in candidate_trim_columns[column_group]:
            empty_ratio = dataframe[column].isna().sum() / len(dataframe) * 100
            NAN_counts.append(empty_ratio)
            column_NaN_counts[column_group] = NAN_counts

    delta_NaN_columns = {}

    for column_group in column_NaN_counts.keys():
        delta_NaN_counts = [] 
        for i in range(0, len(column_NaN_counts[column_group])):
            if i < len(column_NaN_counts[column_group]) - 1:
                delta_NaN_counts.append(column_NaN_counts[column_group][i+1] - column_NaN_counts[column_group][i])
            else:
                continue
        delta_NaN_columns[column_group] = delta_NaN_counts

    return (column_NaN_counts, delta_NaN_columns, candidate_trim_columns)

In [47]:
def make_sparseness_graphs(column_NaN_ratio: dict):
    for column_group in column_NaN_ratio:
        x = [i for i in range(0, len(column_NaN_ratio[column_group]))]
        y = column_NaN_ratio[column_group]

        plt.bar(x, y, color="grey")
        plt.xlabel("Columns")
        plt.ylabel("Sparsity (%)")
        plt.title("Sparsity of {} columns".format(column_group))
        plt.ylim(column_NaN_ratio[column_group][0] - 25, 100)
        #plt.show() #enabling this line will result in saving blank images as plt.show() empties the current plots to make new ones
        #plt.savefig("Figures/Sparsity of" + " {} columns.png".format(column_group).replace("/", "-"))

In [48]:
def get_max_delta_indices(delta_NaN_columns: dict):
    """This function takes in a dictionary and populates a list with indices of the highest increase in sparseness"""
    indices_of_max_columns = []
    for group in delta_NaN_columns.keys():
        maximum = max(delta_NaN_columns[group])
        indices_of_max_columns.append(delta_NaN_columns[group].index(maximum))

    return indices_of_max_columns

In [49]:
def drop_columns(dataframe, candidate_trim_columns: dict, cutoff_indices: list, cutoff_ratio: dict):
    """This function takes in a dataframe, iterates over candidate columns and drops the column if a set of cutoff conditions are met"""
    #based on the index of the max, or if sparseness is above 90%, drop the column
    drop_columns = []

    for i, column_group in enumerate(candidate_trim_columns.keys()):
        for j, column in enumerate(candidate_trim_columns[column_group]):
            if int(column.split(":")[1]) > cutoff_indices[i] and float(cutoff_ratio[column_group][j]) > 90:
                drop_columns.append(column)
            else:
                continue

    for col in drop_columns:
        if "award_contract/val_total:" in col or "object_descr/title:" in col or "ac_criterion" in col:
            drop_columns.remove(col)
        else:
            continue

    dataframe = dataframe.drop(columns = drop_columns, axis = 0)
    return dataframe

In [50]:
column_groups = get_column_groups(xpath_expressions, "multiple")
column_NaN_ratio, delta_NaN_columns, candidate_trim_columns = check_sparsity_dataframe(df, column_groups)
indices_of_max_columns = get_max_delta_indices(delta_NaN_columns) #running twice on an altered version of the df breaks it
df = drop_columns(df, candidate_trim_columns, indices_of_max_columns, column_NaN_ratio)

In [ ]:
val_total_col = []
for index in tqdm(range(0, len(df))):
    row_val = 0
    for col in df.columns:
        cell_value = df[col].iloc[index]
        if "award_contract/val_total:" in col and pd.isna(cell_value) == False:
            money_sequence = cell_value.split(" ")
            for value in money_sequence:
                col_val = int(value.split(".")[0])
                row_val += col_val
        else:
            continue
    if row_val != 0:
        val_total_col.append(row_val)
    else:
        val_total_col.append(np.nan)

In [ ]:
#with open("Data/val_total_col_2021.pickle", "wb") as f:
#    pickle.dump(val_total_col, f)

In [56]:
with open("Data/val_total_col_2021.pickle", "rb") as f:
    val_total_col = pickle.load(f)

In [57]:
#with open("Data/df_data_2021_until_money.pickle", "wb") as f:
#    pickle.dump(df, f)

In [97]:
with open("Data/df_data_2021_until_money.pickle", "rb") as f:
    df = pickle.load(f)

In [98]:
val_total_drop_cols = [col for col in df.columns if "award_contract/val_total:" in col]
df = df.drop(labels=val_total_drop_cols, axis = 1)
df["award_contract/val_total: 0"] = val_total_col

In [99]:
df['award_contract/val_total: 0'] = df['award_contract/val_total: 0'].astype(float)

couple of approaches to getting the cutoffpoint: <br>
1) get the maximum increase in NaN values
2) define a predetermined min and max value

Lots of differences found in the currency within the same tender, unfortunately. The total amount this occurs = 4996 or 0.7%.

The values do seem to differ from object_contract/val_total to award_contract/val_total to award_contract/val_estimated_total currency columns <br>lets compare similar groups so the award contract currency columns. 

Transform all prices to euro currency for uniformity. This requires a dataset of exchange rates, possibly per day. This seems crazy expensive computationally. Because days are almost never present, use the average per year.

In [100]:
#Check the format of the dates in dataframe
currency_exchange_file_path = r"Data\exchange_rates_ECB.csv"
df_currency_exchange = pd.read_csv(filepath_or_buffer=currency_exchange_file_path)

In [101]:
currency_columns = [column for column in df.columns if "currency" in column]
rename_currency_columns = [col.split(".")[2] if len(col.split(".")) > 1 else col for col in df_currency_exchange.columns] #get the currency acronyms like 'EUR'
df_currency_exchange.columns = rename_currency_columns #rename the columns
df_currency_exchange = df_currency_exchange.drop(labels = [i for i in range(0, len(df_currency_exchange)) if i != 21 and i != 22], axis = 'index').reset_index() #drop exchange rates of irrelevant years and reset index
df_currency_exchange = df_currency_exchange.drop(labels = [col for col in df_currency_exchange.columns if df_currency_exchange[col].isnull().any()], axis = 1) #drop columns if they contain NaN

#dropping GIP, ERN, COP, AFN, OMR, GMD, ETB, FKP, PAB, RSD, USN, CYP, EEK, EGP, LTL, MTL
missing_currencies = {"MKD": [61.6404, 61.602], 
                      "MDL": [19.676, 20.8748],
                      "AMD": [555.555, 597.79],
                      "ALL": [123.7864, 122.4619],
                      "FJD": [2.4469, 2.3048],
                      "XOF": [656.0955, 655.9288],
                      "BDD": [2.2849 , 2.3676],
                      "MMK": [1570.00, 1905.74],
                      "SKK": [10.491, 10.1485],
                      "EUR": [1, 1]
                      }

for column in missing_currencies.keys():
    df_currency_exchange[column] = missing_currencies[column]

NaN_currency_columns = [currency for currency in df_currency_exchange.columns if df_currency_exchange[currency].isnull().any()]

#lets take a look at the unique values for currency
currencies = []

for column in currency_columns:
    for currency in df[column].value_counts().keys().tolist():
        currencies.append(currency)

currencies = list(OrderedSet(currencies))

missing_currency_columns = [col for col in currencies if col not in rename_currency_columns]

The dataframe has missing values and not all currencies are represented in the table (stupid ECB!). Where to find the remaining ones? <br>
found one: https://www.exchangerates.org.uk <br>
also, some columns have missing values --> see variable NaN_currency_columns

In [102]:
#dropping GIP, ERN, COP, AFN, OMR, GMD, ETB, FKP, PAB, RSD, USN, CYP, EEK, EGP, LTL, MTL, "CYP" because they occur too infrequently and could not find reliable sources for them
#for col in currency_columns:
#   print(df[col].value_counts())

As the code above shows, some rows with have values for one or more of the money columns. Being text, any of these columns may contain one or more numbers. As such, if multiple numbers are mentioned, they seperate transformed to an integer and then added to make a total.

In [103]:
def check_currency(df, money_row_curs: list, money_list: dict, index_money_row: int, money_row: dict, money_key, currency_key, country_to_currency_mapping = country_to_currency_mapping):
    
    country = df["country"].iloc[index_money_row]
    some_curs = [currency for currency in money_row_curs if currency != None and len(currency) == 3]
    if len(some_curs) > 0:
        for k in range(0, len(some_curs)):
            if some_curs[0] != some_curs[len(some_curs)-1-k]:
                #they differ, I dont know what to do with it so set the money amount to None
                money_list[index_money_row][money_key] = None
                money_list[index_money_row][currency_key] = None
            else:
                #they do not differ, assign the currency that was found to be similar in the whole row
                assigned_currency = some_curs[0]
                money_list[index_money_row][currency_key] = assigned_currency
    elif df['country'].iloc[money_row["index"]] in [country for country in pays_with_euro.keys() if pays_with_euro[country] == 1]:
        #all currencies were none check if it is an euro_zone country
        money_list[index_money_row][currency_key] = 'EUR'
    
    elif country in country_to_currency_mapping.keys():
        assigned_currency = country_to_currency_mapping[country]
        money_list[index_money_row][currency_key] = assigned_currency
    else:
        return index_money_row

In [104]:
def exchange_currency(money_list: list, index_money_row: int, money_key: int, df_currency_exchange, money_row: dict, currency: str, year: str):
    if currency != None:
        currency_exchange_year = {2020: 0, 2021: 1}
        conversion_rate = df_currency_exchange[currency][currency_exchange_year[year]]
        total_amount = 0

        for money_piece in str(money_row[money_key]).split(" "):
            if len(re.findall("([\d.]*\d+)", money_piece)) > 0:
                money_piece = money_piece.split(".")[0] #just keep the whole number rather than what is after the comma
                total_amount += int(float(re.findall("([\d.]*\d+)", money_piece)[0]))

        total_new_amount = int(total_amount / conversion_rate)
        money_list[index_money_row][money_key] = total_new_amount

In [105]:
money_columns = [col for col in df.columns if "val" in col and len(col.split("[")) < 2]
currency_columns

['object_contract/val_estimated_total[@currency]',
 'object_contract/val_object[@currency]',
 'object_contract/val_total[@currency]',
 'award_contract/val_total[@currency]',
 'award_contract/val_estimated_total[@currency]']

In [ ]:
rows_with_money = []

for i in tqdm(range(0, len(df))):
    money_row = {}
    for col in money_columns:
        currency_col = col.split(":")[0] + "[@currency]"
        if df[col][i]!= None and pd.isna(df[col][i]) != True:
            money_row["index"] = i
            money_row[col] = df[col][i]
            money_row[currency_col] = df[currency_col][i]
    
    if len(money_row) > 0:
        rows_with_money.append(money_row)

rows_no_currencies = []

for index_money_row, money_row in tqdm(enumerate(rows_with_money)):
    money_row_curs = [money_row[column_name] for column_name in money_row.keys() if "@" in column_name]
    money_row_cur_cols = [column_name for column_name in money_row.keys() if "@" in column_name]

    for index_currency, currency in enumerate(money_row_curs):
        money_key = list(money_row.keys())[index_currency*2+1]
        currency_key = list(money_row.keys())[index_currency*2+2]

        if currency == np:
            index = check_currency(df = df, money_row_curs=money_row_curs, money_list=rows_with_money, index_money_row=index_money_row, money_row = money_row, money_key = money_key, currency_key=currency_key)
            exchange_currency(money_list=rows_with_money, index_money_row=index_money_row, df_currency_exchange=df_currency_exchange, money_row=money_row, currency=currency, money_key=money_key, year = 2020)
            if index != None:
                rows_no_currencies.append(rows_with_money[index]["index"])
            #exchange_currency()

        elif currency != None and currency in df_currency_exchange.columns:
            exchange_currency(money_list=rows_with_money, index_money_row=index_money_row, df_currency_exchange=df_currency_exchange, money_row=money_row, currency=currency, money_key=money_key, year = 2020)
        
        elif currency != None and currency not in df_currency_exchange.columns:
            #currency unknown, set value to None
            rows_with_money[index_money_row][money_key] = None

rows_no_currencies = list(OrderedSet(rows_no_currencies))
len(rows_no_currencies)

In [ ]:
len(df.loc[(pd.isna(df[money_columns[0]]) == False) |
       (pd.isna(df[money_columns[1]]) == False) |
       (pd.isna(df[money_columns[2]]) == False) |
       (pd.isna(df[money_columns[3]]) == False) |
       (pd.isna(df[money_columns[4]]) == False)])

the length of the rows_no_currencies is quite big, 6692 to be precise. Lets take a look at the data. A lot of the countries in the list have euro as their currency so it goes without saying the currency is in euro so it can be kept.
However, lets make a list comparing them. of the 6692, 5028 of them belong to tenders from a country where they use the euro.

info old df
length of rows_no_currencies = 1664<br>
length of dataframe = 643554<br>
len no_money_values list = 271116 <br>
dimensions df_data = (372438, 47) <br>

info new df
length of rows_no_currencies = 106007<br>
length of dataframe before = 643554<br>
len no_money_values list = 0<br>
dimensions df_data = <br>

so it seems that the change made in contract_award/val_total just introduces more rows which do have a target variable but no currencies, let's take a look at these rows. just lots of countries which are not euro zone and are also not in the currency exchange list (as they have no currency), but the countries can be mapped to currencies and as such translated in the end. Let's make one by hand. taking this into account, there should be 99850 cases less in the no_currency list. It was previously 106007 so this should results in 6157 cases. Turns out I changed NaN with 0 which is why every row was counted as a row with a money value which is obviously not true. Changing that back to Nan rather than 0 resulted in: 372276 rows in rows_with_money and 1024 cases of no currencies.

In [ ]:
#lets drop the non money rows but keep the original indices and add them as a seperate column so we can add shit to it later if need be
df_money = pd.DataFrame.from_records(rows_with_money, index = [rows_with_money[index]["index"] for index in range(0, len(rows_with_money))])
no_money_values = list(set(df.index.tolist()) - set(df_money["index"].to_list()))
len(df_money)

In [ ]:
len(df)

In [ ]:
df["awarded_contract"].value_counts()

In [ ]:
len(df.loc[df["awarded_contract"] == "AWARDED_CONTRACT"])

In [ ]:
df = df.drop(labels = no_money_values)
df[money_columns] = df_money[money_columns]
df[currency_columns] = df_money[currency_columns]

#now that all the money columns have been translated to euro values, lets drop the currency columns and reset the index
df = df.drop(currency_columns, axis = 1)
df = df.reset_index()
df = df.rename(columns={"index": "original index"})

#lets drop the rows of which is it not known if the tender was awarded
print("The amount of awarded contracts is {} tenders".format(len(df.loc[df["awarded_contract"] == "AWARDED_CONTRACT"])))

In [90]:
df = df.fillna(value = np.nan)

Of the values of column renewal, existing values fill up 28.150888050801022%
Of the values of column procedure, existing values fill up 89.01540792315379%
Of the values of column recurrent, existing values fill up 28.163244474529648%
Of the values of column eu_fund, existing values fill up 99.05204740568824%
Of the values of column framework or dynamic purchasing, existing values fill up 17.280458584491075%

it appears that the boolean column values get lost after dropping the awarded contract, let's take a closer look.
We know that the renewal and recurrent values are there before dropping != AWARDED_CONTRACT, but let's look at the split between
renewal and recurrent for Awarded and not awarded. Turns out all the rows where recurrent is not Nan, are those rows where there is no value for awarded_contract. 
This is super strange, cant really seem to figure it out. Let's check if this also holds true for the renewal column. 

In [ ]:
len(df.loc[(df["awarded_contract"] != "NO_AWARDED_CONTRACT") & 
                 (df["awarded_contract"] != "AWARDED_CONTRACT") &
                 (pd.isna(df["recurrent"]) != True)]) == df["recurrent"].value_counts().sum()

len(df.loc[(df["awarded_contract"] != "NO_AWARDED_CONTRACT") & 
                 (df["awarded_contract"] != "AWARDED_CONTRACT") &
                 (pd.isna(df["renewal"]) != True)]) == df["renewal"].value_counts().sum()

In [25]:
indices_not_awarded = [index for index in range(0, len(df)) if df["awarded_contract"].iloc[index] != "AWARDED_CONTRACT"]
df = df.drop(labels=indices_not_awarded).reset_index(drop = True)

In [26]:
df = df.drop(columns = ["awarded_contract"])

the code below takes in the list of lists with each row from the df containing unknown currencies, this can be either None or some currency not incorporated.
as te results show, only 42 of the 7669 actually contain currencies not accounted for in this piece of research, lets drop these rows. The other rows, if they have None, they have
listed a currency in one or more of the currency columns, of which they are identical. As such, we can use the difference between these indices and use whatever currency is present in the other
columns as the main currency for the whole row

the code above shows how, of the rows where a currency was unknown (not in list or None), do have currencies in other columns specified which we can use. However, this leaves 42 cases of the indices_unknown_currencies cases unaccounted for, lets take a look at them.

lets take a look at the money columns. It seems that the award_contract/val_total is the target variable, or, lowest offer/highest offer. Only 6500 rows of the 239413 rows have no value for award_contract/val total !!! that is great. It seems that object_contract/val_total is the vlaue of the procurement rather than the tender so it can be bigger.

seems that only in 5468 of 239413 cases or 2.23% of cases have a higher contract value than the object value.

In [ ]:
#check the count of rows having a value for award_contract/val_total
indices_no_target_var = []
indices_target_var = []

for i in range(0, len(df)):
    if pd.isna(df["award_contract/val_total: 0"][i]) == True:
        indices_no_target_var.append(i)
    elif pd.isna(df["award_contract/val_total: 0"][i]) != True:
        indices_target_var.append(i)
    else:
        continue
print(len(indices_no_target_var), len(indices_target_var))

it looks like almost all cases have a lowest/highest offer instead of the award_contract/val_total which makes sense as that is an alternative in the original tender documents. Lets take a closer look

In [28]:
def add_offers(offer: str):

    offer_list = offer.split(".")
    final_offer = 0
    for offer in offer_list:
        final_offer += int(offer)

    return final_offer

In [ ]:
indices_high_low_offer = {}
indices_same_high_low = []
for i, index in enumerate(indices_no_target_var):
    low_offer = df["lowest offer"].iloc[index]
    high_offer = df["highest offer"].iloc[index]

    if pd.isna(low_offer) != True and pd.isna(high_offer) != True:
        low_offer = add_offers(str(low_offer))
        high_offer = add_offers(str(high_offer))
        indices_high_low_offer[index] = [low_offer, high_offer]
        if low_offer == high_offer:
            indices_same_high_low.append(index)
        else:
            continue
    else:
        continue

print(len(indices_high_low_offer),len(indices_same_high_low))
#5832 recorded cases, not as much as hoped but still we can incorporate it. let's check how often the offers are the same.
#offers are the same 2213 times out of 5832, initially thinking the tender only received one bid but this is actually not true. Regardless, pick the number of same bids
#as the target variable. What to do with the others though? Because of MEAT, lowest offer is not always the right one. we can filter in combination with the quality, cost
#and price criterion. To do that, lets unify the current df with the dictionary containing the information on costs. We also need the weights from price which we dont have...

In [30]:
no_target_variable_and_offer = set(indices_no_target_var) - set(indices_high_low_offer)

THE CELL BELOW IS A PIECE OF CODE CHECKING FOR PATTERNS IN THE REMAINING CASES BUT UNFORTUNATELY NOTHING RELIABLE CAN BE MADE OF THESE CASES, THEREFORE, THE ROWS ARE DROPPED

In [31]:
df = df.drop(labels = no_target_variable_and_offer).reset_index(drop=True)

In [32]:
#lets drop the highest and lowest offer column

In [33]:
#lets check for the remaining cases where there is no low or high offer
#no_target_variable_and_offer = list(set(indices_no_target_var) - set(indices_high_low_offer.keys()))

#how many of the cases have an award_contract/val_estimated_total
#indices_award_contract_val_estimated_total = []

#for i, index in enumerate(no_target_variable_and_offer):
#    value = df["award_contract/val_estimated_total"].iloc[int(index)]
#    if pd.isna(value) == False and value != None and value > 0:
#        indices_award_contract_val_estimated_total.append(index)
#    else:
#        continue
    
#print(len(indices_award_contract_val_estimated_total))
#let's look at the row which do not have a target variable.
#df.iloc[no_target_variable_and_offer][:100]

CODE BELOW WAS USED TO DO SOME RESEARCH TO MATCH UP THE AC_COST COLUMNS WITH THE DF_DATA

In [ ]:
placed_Here_to_interupt!
#let's remove all of the key:value pairs which are stored in the list no_money_values
for index in no_money_values:
    del data_dict_2020_ac_cost[index]

for index in indices_not_awarded:
    if index in data_dict_2020_ac_cost.keys():
        del data_dict_2020_ac_cost[index]
    else:
        continue

for index in no_target_variable_and_offer:
    if index in data_dict_2020_ac_cost.keys():
        del data_dict_2020_ac_cost[index]
    else:
        continue

#let's reduce the amount of rows further by removing empty rows
keys_for_delete = []
for key in data_dict_2020_ac_cost.keys():
    if (data_dict_2020_ac_cost[key]["ac_cost/ac_criterion: 0"] and data_dict_2020_ac_cost[key]["ac_cost/ac_weighting: 0"]) == None:
        keys_for_delete.append(key)

for key in keys_for_delete:
    del data_dict_2020_ac_cost[key]

len(data_dict_2020_ac_cost)

In [36]:
#we will no longer use any of the money columns other than the target variable from here on out so let's drop these columns
df = df.drop(labels = ["object_contract/val_estimated_total: 0", "object_contract/val_object: 0", "object_contract/val_total: 0", "award_contract/val_estimated_total"], axis = 1)

In [37]:
with open("Data/df_data_2021_up_until_time_columns.pickle", "wb") as f:
    pickle.dump(df, f)

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING OF TIME COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [38]:
#lets save the work we have as of now so we dont have to redo all the manipulations when we do something that doesnt make sense
with open("Data/df_data_2021_up_until_time_columns.pickle", "rb") as f:
    df = pickle.load(f)

#with open('Data/currency_columns_2021.pickle', 'rb') as f:
#    currency_columns = pickle.load(f)

#with open('Data/money_columns.pickle', 'rb') as f:
#    money_columns = pickle.load(f)

#with open('Data/time_columns.pickle', 'rb') as f:
#    time_columns = pickle.load(f)

In [39]:
def convert_to_days(index: int, time_amount: int, average_days_per_month = 30.437):
    """"This function calculates months to days"""
    time_unit = df["duration type"][index]
    if time_unit == 'MONTH':
        amount_in_days = int(int(time_amount) * average_days_per_month)
    else:
        amount_in_days = int(time_amount)
    return amount_in_days

In [40]:
duration_column = "object_descr/duration: 0"
start_date_column = "object_descr/date_start: 0"
end_date_column = "object_descr/date_end: 0"
duration_column_calculated = []
strange_days_count = []
time_indices = []

for index in range(0, len(df)):
    duration = df[duration_column][index]
    start = df[start_date_column][index]
    end = df[end_date_column][index]
    
    if pd.isna(duration) != True and pd.isna(start) == True and pd.isna(end) == True:
        duration_column_calculated.append(convert_to_days(index, duration))
        time_indices.append(index)

    elif pd.isna(duration) == True and pd.isna(start) != True and pd.isna(end) != True:
        start_date = datetime.strptime(str(start), "%Y-%m-%d")
        end_date = datetime.strptime(str(end),"%Y-%m-%d")
        delta_days = (end_date - start_date).days
        duration_column_calculated.append(delta_days)
        time_indices.append(index)

    elif pd.isna(duration) != True and (pd.isna(start) != True or pd.isna(end) != True):
        duration_column_calculated.append(convert_to_days(index, duration))
        time_indices.append(index)

    else:
        duration_column_calculated.append(None)
        strange_days_count.append(index)

In [41]:
df["duration"] = duration_column_calculated

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#how many of the rows with a duration do not have a duration type?
type_month_count = list(df['duration type'].value_counts().values)[0]
type_day_count = list(df['duration type'].value_counts().values)[1]
print("{}% of cases with a duration type is of time type 'month'. But {}% does not have a time type at all".format(int(type_month_count / (type_month_count + type_day_count) * 100), 
                                                                                                                  int(100 - (type_month_count+type_day_count) / len(df)* 100)))

df["duration type"].value_counts()

In [ ]:
#we cannot look at the distribution of the months because there is something going on with the data, lets find out what it is
#lets look at those cases where type is days to see the average time when expressed in days such that we can infer what the type should be in case type is None
duration_in_days = df['duration'].loc[df["duration type"] == 'DAY'].values
duration_in_months = df['duration'].loc[df["duration type"] == 'MONTH'].values

outlier_lower_bound_days = int(0.05*len(duration_in_days))
outlier_upper_bound_days = int(0.95*len(duration_in_days))

outlier_lower_bound_months = int(0.05*len(duration_in_months))
outlier_upper_bound_months = int(0.95*len(duration_in_months))

dur_sorted_days = np.sort(duration_in_days)[outlier_lower_bound_days:outlier_upper_bound_days]
dur_sorted_months = np.sort(duration_in_months)[outlier_lower_bound_months: outlier_upper_bound_months]
duration_arrays = {'days': dur_sorted_days, 'months': dur_sorted_months}

In [ ]:
df['duration'].loc[df["duration type"] == 'MONTH'];

The boxplots clearly show a longer overall object duration for those tenders which are expressed in months, this seems to make sense as you wouldnt say 365 days if you can just note 12 months. However, lets also try to get rid of the top and bottom 5% of outliers and see if this makes them more the same. Even with removing the outliers, the amount in months seems to be much higher and with a longe tail on the longer duration side. The distribution with original in days has much more sightings outside of the normal distribution, indicating it is less standardized.

In [ ]:
#lets make a boxplot to see the distribution and match that with what we find in the dataset
for key in duration_arrays.keys():
    duration_boxplot, ax1 = plt.subplots()
    ax1.set_title("Distribution of duration in days (original={})".format(key))
    ax1.boxplot(duration_arrays[key])
    #plt.show()
    plt.savefig("Figures/Distribution of duration in days (original={}).png".format(key))

In [ ]:
for col in time_columns:
    print(col, ": ", df[col].isna().sum(), " = ", (df[col].isna().sum() / len(df) * 100), "%")

In [ ]:
len(df) - df["duration"].isna().sum()

In [ ]:
#duration column is 90% empty, might have to drop it. with 87% of cases belonging to month, we just randomly assign according to this distribution
options = ["MONTH", "DAY"]

for i, row in enumerate(range(0, len(df))):
    if pd.isna(df["duration"].iloc[i]) == False and df["duration type"].iloc[i] == None:
        df.at[i, "duration type"] = random.choices(options, weights = (0.87, 0.13), k=1)[0]
    else:
        continue

In [ ]:
[col for col in time_columns if col != "duration"]

In [ ]:
#lets drop the duration column and other time columns, except date_conclusion_contract
df = df.drop(labels = [col for col in time_columns if col != "duration" and col != "date_conclusion_contract"], axis = 1)
df = df.drop(labels = "duration type", axis = 1)

In [ ]:
with open("Data/df_data_2021_up_until_categorical.pickle", "wb") as f:
    pickle.dump(df, f)

---------------------------------------------------------------------------------------------------------------------------------------
OLD CODE FOR PREPROCESSING OF TIME COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

------over 1621 or 0.4% of the original, or  do have either a start_date or an end date, but no duration. How do we determine the duration of the contract? maybe it can be retrieved from the name in the xml_file_list. Even though they are the same length, lets check a couple of them, just to be sure. IT WORKS! However, date of contract notice publication is not the same as the start date of the contract,  by approximation we could use the different max days for a given tender procedure but that is still an approximation and the same size for which this occurs is too small, making it not worth my while really. ------

In [ ]:
#This piece of code was used to determine the structure of the xml file names
#file_list_dates = []
#for file_name in xml_files_list_2020:
#    file_list_dates.append(file_name.split("/")[2])
#
#set(file_list_dates)

question is, is date_conclusion_contract or date_dispatch_notice relevant? if anything specific is to happen influencing the tender, it is unlikely to be a pattern that can be detected by the algorithm on a date basis. Per year might differ but thats already processed in other elements. It is relevant to order the time data. lets check the publication date.
seems like date_dispatch_notice is when the notice got released as is rather than when it was called for tender as the dates are at a later point in time than the conclusion of the tender. lets look at the values. Date of the dispatch notice is always there, probably because it is automated by the system.

by eyeballing the data, it does not seem that there is a column which can be conveniently and accurately used as the date indicator to establish a time series. <br>
date_conclusion_contract seems like the best candidate with actual data, however it still has 31% missing data so any strategy would have a severe limitation. Although the date dispatch_notice is not truly reflecting the time at which the contract was published or concluded, it is always present and most consistent and will therefore be used to create the time split.

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING OF CATEGORICAL COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
with open("Data/df_data_2021_up_until_categorical.pickle", "rb") as f:
    df = pickle.load(f)

In [ ]:
#lets unify the ca_type and ca_type_other column
ca_type_col = []
for index in range(0, len(df)):
    ca_type = df["ca_type"].iloc[index]
    ca_type_other = df["ca_type_other"].iloc[index]
    if pd.isna(ca_type) == False:
        ca_type_col.append(ca_type)
    elif pd.isna(ca_type_other) == False:
        ca_type_col.append(ca_type_other)
    else:
        ca_type_col.append(None)

ca_activity_col = []
for index in range(0, len(df)):
    ca_activity = df["ca_activity"].iloc[index]
    ca_activity_other = df["ca_activity_other"].iloc[index]
    if pd.isna(ca_activity) == False:
        ca_activity_col.append(ca_activity)
    elif pd.isna(ca_activity_other) == False:
        ca_activity_col.append(ca_activity_other)
    else:
        ca_activity_col.append(None)

In [ ]:
df["contracting authority type"] = ca_type_col
df["contracting authority activity"] = ca_activity_col

In [ ]:
#now lets drop the old columns
df = df.drop(labels = ["ca_type", "ca_activity", "ca_type_other", "ca_activity_other"], axis = 1)

In [ ]:
#lets check the values of the newly made columns. That is looking pretty decent
print(df["contracting authority type"].value_counts(), "\n\n", df["contracting authority activity"].value_counts())

contracting authority type
REGIONAL_AUTHORITY    72804
BODY_PUBLIC           46248
CA_TYPE_OTHER         37727
MINISTRY              22751
REGIONAL_AGENCY        5659
NATIONAL_AGENCY        5357
EU_INSTITUTION         1354
Name: count, dtype: int64 

 contracting authority activity
GENERAL_PUBLIC_SERVICES            78728
CA_ACTIVITY_OTHER                  33217
HEALTH                             32434
EDUCATION                          14012
HOUSING_AND_COMMUNITY_AMENITIES     7674
ENVIRONMENT                         6097
PUBLIC_ORDER_AND_SAFETY             5614
ECONOMIC_AND_FINANCIAL_AFFAIRS      4951
DEFENCE                             4133
SOCIAL_PROTECTION                   2818
RECREATION_CULTURE_AND_RELIGION     2175
Name: count, dtype: int64

lets look at ac_weighting, ac_cost and ac_price, what does it mean and what kind of values are we working with? Looks like in case of MEAT, people use criterion, weightings, and ac_price and ac_price_weighting. if not they use costs. lets see if this is true.

lets look at ac_criterion first. it appears that there are 90937 rows with a value for ac_criterin, of which 32030 are unique. This is way to much to combine into a categorical variable, but again they may be unique
just because they are all in different languages, it is likely that they will share a similar meaning, might have to vectorize these with a language model as they will get similar vectors for similar topics.
with that in mind, lets look at the length, amount and of words of these cells to get an idea of the content. Another option would be to do topic modelling on the criteria column, classify each row to one of the topics and then one hot encode, and put the weights for that criteria under the column. For now, lets try and continue with the price column first.

In [ ]:
print(len(df["ac_criterion: 0"].unique()), len(df) - df["ac_criterion: 0"].isna().sum()) 

In [ ]:
#lets replace all None values with NaN, should have done this a long time ago but alright
df = df.fillna(value = np.nan)

In [ ]:
string_length_ac_criterion = []
word_length_ac_criterion = []

for i, row in enumerate(range(0,len(df))):
    if pd.isna(df["ac_criterion: 0"].iloc[i]) != True:
        cell = str(df["ac_criterion: 0"].iloc[i])
        string_length_ac_criterion.append(len(cell))
        word_length_ac_criterion.append(len(cell.split(" ")))

string_length_ac_criterion = np.array(string_length_ac_criterion)
word_length_ac_criterion = np.array(word_length_ac_criterion)

In [ ]:
len(string_length_ac_criterion)

In [ ]:
print(pd.DataFrame(string_length_ac_criterion).describe(), "\n\n", pd.DataFrame(word_length_ac_criterion).describe())

In [ ]:
#lets drop the empty column
df = df.drop(labels = ["ac_price: 0"], axis = 1)

In [ ]:
#lets look at the kind of values we are working with and make them homogeen
indices_got_price_weighting = [i for i in range(0, len(df)) if pd.isna(df["ac_price_weighting: 0"].iloc[i]) == False]

In [ ]:
def find_numbers(value: str):
    numbers_array = re.findall(r'[0-9]+', value)
    return numbers_array

In [ ]:
def find_only_numbers(value: str):
    numbers_array = re.findall(r'^[0-9]{1,3}', value)
    return numbers_array

In [ ]:
def find_letters(value: str):
    letter_array = re.findall(r'[a-zA-Z]+', value)
    return letter_array

In [ ]:
count = 0
for i in range(0,len(df)):
    value = str(df.iloc[i])
    numbers_list = find_numbers(value)
    if len(numbers_list) > 0:
        if "%" in value or "-" in value or "/" in value or "&" in value or "*" in value:
            #print("{}: something with %, - or /: {}".format(i, value))
            value = int(numbers_list[0])
           # print(value)
            if value > 100:
                #print("{} can't have more than 100%: {}".format(i, value))
                value = 0
        elif "," in value or "." in value:
            if int(numbers_list[0]) == 0:
                #print("{}: something with 0.x or 0,x: {}".format(i, value))
                if int(numbers_list[1]) > 10:
                    value = int(numbers_list[1])
                    #print(value)
                else:
                    value = int(numbers_list[1][0]) * 10
                    #print(value)
            else:
                #print("{}: something with XX.X or XX,X: {}".format(i, value))
                value = int(numbers_list[0])
                #print(value)
        elif len(re.findall(r"^[0-9]+", value)) > 0:
            if int(re.findall(r"^[0-9]+", value)[0]) < 101:
                #print("value at {} is {}".format(i, int(re.findall(r"^[0-9]+", value)[0])))
                value = int(re.findall(r"^[0-9]+", value)[0])
            else:
                #print("value at {} is {} and will be set to 0". format(i, value))
                value = 0
        elif len(re.findall(r"[0-9]+$", value)) > 0:
            if int(re.findall(r"[0-9]+$", value)[0]) < 101:
                #print("value at {} is {}".format(i, int(re.findall(r"[0-9]+$", value)[0])))
                value = int(re.findall(r"[0-9]+$", value)[0])
            else:
                #print("value at {} is {} and will be set to 0". format(i, value))
                value = 0
        else:
            value = 0
    else:
        value = 0
        
    df.at[i, "ac_price_weighting: 0"] = value


In [ ]:
#let's drop ac_cost: 0 column
df = df.drop(labels = ["ac_cost: 0"], axis = 1)

UNFORUTNATELY, TOPIC MODELLING ALL THE CRITERIA, ASSIGNING AS CATEGORICAL VARIABLES AND PLACING WEIGHTS WITHIN THE CELL IN THE RESPECTIVE COLUMN IS NOT FEASIBLE FOR THIS THESIS. THUS, WHAT WE CAN DO IS ADD THE TEXT OF ALL CRITERIA COLUMNS AND MAKE DOCUMENT VECTORS OUT OF THEM. WE DO HAVE TO DROP THE WEIGHTS AS A RESULTS BECAUSE IT CAN NOT BE LINKED TO THE CRITERIA IN AND OF ITSELF. IN EXTENSION OF THIS THOUGHT PROCESS, WE WILL DO THE SAME TO OBJECT_DESCR/TITLE

In [ ]:
ac_criterion_cols = [col for col in df.columns if "ac_criterion" in col and "ac_cost" not in col]
ac_weights_columns = [col for col in df.columns if "ac_weighting" in col]

ac_cost_criterion_cols = [col for col in df.columns if "ac_cost/ac_criterion:" in col]
object_descr_title_cols = [col for col in df.columns if "object_descr/title:" in col]

df = df.drop(labels = ac_weights_columns, axis = 1)

aggregate_groups = [object_descr_title_cols, ac_cost_criterion_cols, ac_criterion_cols] 

In [ ]:
total = []

for column_group in aggregate_groups:
    aggregated_data = []
    for i in range(0, len(df)):
        info_per_row = ""
        for col in column_group:
            if pd.isna(df[col].iloc[i]) == False:
                if str(df[col].iloc[i]) not in info_per_row:
                    info_per_row += df[col].iloc[i]
                else:
                    continue
            else:
                continue
        aggregated_data.append(info_per_row)
    total.append(aggregated_data)

df["object descr titles"] = total[0]
df["ac cost criteria"] = total[1]
df["ac criteria"] = total[2]

In [ ]:
#now lets drop all the columns we cleaned up here
df = df.drop(labels=ac_cost_criterion_cols, axis = 1)
df = df.drop(labels=object_descr_title_cols, axis = 1)
df = df.drop(labels=ac_criterion_cols, axis = 1)

#interesting enough there are multiple weighthing columns for ac_criterion but no additional columns for the criterion itself... what is going on? It is also not in the original df. Lets take a look in the extraction notebook. Apparently, it is in the original data_dict but got lost somewhere along the way, something to keep in mind when processing the data for 2021. For now, all text is aggregated in the one column to "document vectorize" later on.

In [ ]:
with open("Data/df_data_2021_up_until_boolean.pickle", "wb") as f:
    pickle.dump(df, f)

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING OF BOOLEAN COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
df = pd.read_pickle("Data/df_data_2021_up_until_boolean.pickle")

results from the value_counts performed below shows some interesting stuff. <br>
(1) in column joint_procurement_involved it only notes it is was yes. As such, we will assume that all other cases did not involve joint procurement. <br>
(2) same thing with central_purchasing as with (1) <br>
(3) Probably best to take care of the null values by introducing a third category, being neither of the options. This may be applied to procedure (if not all cases have a procedure), renewal, recurrent, framework or dynamic purchasing. 

In [ ]:
#lets look at the renewal, procedure, recurrent, join_procurement_involved, central_purchasing, eu_fund and framework and dynamic purchasing
mixed_columns = ["renewal", "procedure", "recurrent", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing"]
for col in mixed_columns:
    print(df[col].value_counts())

In [ ]:
for index in range(0, len(df)):
    central_purchasing = df["central_purchasing"].iloc[index]
    if pd.isna(central_purchasing) == True:
        df.at[index, "central_purchasing"] = "NO_CENTRAL_PURCHASING"
    else:
        continue

for index in range(0, len(df)):
    joint_procurement = df["joint_procurement_involved"].iloc[index]
    if pd.isna(joint_procurement) == True:
         df.at[index, "joint_procurement_involved"] = "NO_JOINT_PROCUREMENT_INVOLVED"
    else:
        continue

In [ ]:
for index in tqdm(range(0, len(df))):
    joint_procurement = df["framework or dynamic purchasing"].iloc[index]
    if pd.isna(joint_procurement) == True:
         df.at[index, "framework or dynamic purchasing"] = "no framework or dynamic purchasing"
    else:
        continue

In [ ]:
#if the procedure is missing, let's impute with open
for index in tqdm(range(0, len(df))):
    procedure = df["procedure"].iloc[index]
    if pd.isna(procedure) == True:
         df.at[index, "procedure"] = "PT_OPEN"
    else:
        continue

In [ ]:
df["framework or dynamic purchasing"].value_counts()

In [ ]:
none_of_the_above_cols = ["renewal", "procedure", "recurrent", "eu_fund", "framework or dynamic purchasing"]

In [ ]:
df["framework or dynamic purchasing"].value_counts()

With the original, unprocessed dataset, these were the stats:
Of the values of column renewal, existing values fill up 36.34116173623348% <br>
Of the values of column procedure, existing values fill up 76.13486980113558% <br>
Of the values of column recurrent, existing values fill up 36.35405886685499% <br>
Of the values of column eu_fund, existing values fill up 84.88969068640705% <br>
Of the values of column framework or dynamic purchasing, existing values fill up 14.45752803960507% <br>
<br>
This is widely different from the processed dataset, where we would expect the numbers to be the same. The fact that they are not tells me one of two things (1) I fucked up, or (2) these values appear to only be present in those tenders which get filtered out. let's check the up_until_time_columns dataset. It appears that the data is lost ariving at the up_until_time_columns so let's check in between there. It still looks good before processing the money columns. Lets filter on the money columns and check again. During the dropping of the no_money indices, the values for renewal and recurrent are lost. TURNS OUT I DID EVERYTHING RIGHT, THERE IS JUST A RELATIONSHIP BETWEEN A CONTRACT NOT BEING AWARDED AND RENEWAL AND RECURRENT BEING EMPTY. PROBABLY BECAUSE OF SOME PROCEDURAL KNOWLEDGE I AM NOT AWARE OF

In [ ]:
for col in none_of_the_above_cols:
    print("Of the values of column {}, existing values fill up {}%".format(col, dfe_counts().values.sum() / len(df)*100))

In [ ]:
#Let's drop the renewal and recurrent columns
df = df.drop(columns = ["recurrent", "renewal"])

In [ ]:
none_of_the_above_cols = [col for col in none_of_the_above_cols if col != "recurrent" and col != "renewal"]
#lets call the new option [column_name]_other
for col in none_of_the_above_cols:
    for index in range(0, len(df)):
        if df[col].iloc[index] == None:
            df.at[index, col] = "{}_other".format(col)
        else:
            continue

In [ ]:
with open("Data/df_data_2021_up_until_remaining.pickle", "wb") as f:
    pickle.dump(df, f)

lets see how many of the rows are actually tenders that were awarded

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING OF REMAINING COLUMNS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
df = pd.read_pickle("Data/df_data_2021_up_until_remaining.pickle")

In [ ]:
#official name of the contracting agency should not matter so also drop it. 
df = df.drop(labels=["official_name"], axis = 1)

In [ ]:
df.head()

In [ ]:
#cant process the weight columns unfortunately so let's gather the columns and remove them
#weight_cols = [col for col in df.columns if "weighting:" in col]
#df = df.drop(columns = weight_cols)

it looks like the same bidders may bid for various items in the contract. Because we can only use one number, we will take the average. First, lets see what the distribution looks like to see if this is a fair representation

In [ ]:
#let's look at the number of tenders recöeived
df["nb_tenders_received"].loc[pd.isna(df["nb_tenders_received"]) == False]

In [ ]:
#let's look at the number of tenders received from SMEs
df["nb_tenders_received_sme"].loc[pd.isna(df["nb_tenders_received_sme"]) == False]

In [ ]:
number_of_bids_col = []

for row in range(0, len(df)):
    if pd.isna(df["nb_tenders_received"].iloc[row]) != True:
        cell = str(df["nb_tenders_received"].iloc[row]).split(".")
        total = 0
        for number in cell:
            total += int(number)
        number_of_bids_col.append(int(total / len(cell)))
    else:
        number_of_bids_col.append(np.nan)

df["nb_tenders_received"] = number_of_bids_col

number_of_bids_sme_col = []

for row in range(0, len(df)):
    if pd.isna(df["nb_tenders_received_sme"].iloc[row]) != True:
        cell = str(df["nb_tenders_received_sme"].iloc[row]).split(".")
        total = 0
        for number in cell:
            total += int(number)
        number_of_bids_sme_col.append(int(total / len(cell)))
    else:
        number_of_bids_sme_col.append(np.nan)

df["nb_tenders_received_sme"] = number_of_bids_sme_col

In [ ]:
df.head()

In [ ]:
df["lowest offer"] = val_total_low_offer

In [ ]:
#we need to unify the low and high offer with value total, assuming the lowest offer was chosen, we can drop the high column, and replace the value in val_total with the lowest 
#offer if val_total == Nan
count = 0
for i in range(0, len(df)):
    if pd.isna(df["lowest offer"].iloc[i]) == False and pd.isna(df["award_contract/val_total: 0"].iloc[i]) == True:
        df.at[i, "award_contract/val_total: 0"] = float(df["lowest offer"].iloc[i])
        count +=1
count

In [ ]:
df = df.drop(columns = [col for col in money_columns if "award_contract/val_total: 0" not in col and "award_contract/val_total: 1" not in col])

In [ ]:
df = df.drop(columns = ["lowest offer", "highest offer"])

In [ ]:
#let's transform the lowest value so we can add it
val_total_low_offer = []
for index in tqdm(range(0, len(df))):
    row_val = 0
    for col in df.columns:
        cell_value = df[col].iloc[index]
        if "lowest offer" in col and pd.isna(cell_value) == False:
            money_sequence = cell_value.split(" ")
            for value in money_sequence:
                col_val = int(value.split(".")[0])
                row_val += col_val
        else:
            continue
    if row_val != 0:
        val_total_low_offer.append(row_val)
    else:
        val_total_low_offer.append(np.nan)

In [ ]:
no_target_variable_list = df["award_contract/val_total: 0"].loc[pd.isna(df["award_contract/val_total: 0"]) == True].index.tolist()
df = df.drop(labels = no_target_variable_list, axis = 0).reset_index(drop = True)

In [ ]:
df.head()

In [ ]:
with open("Data/df_data_2021_part_1.pickle", "wb") as f:
    pickle.dump(df, f)